# 04 · Task 1 summary (offline)

Builds the Task 1 tables and figure from saved `per_example.csv` files. No API calls; the API key is never loaded.

In [1]:
import os, sys, json
from pathlib import Path
ROOT = Path.cwd().resolve()
if not (ROOT / "common").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from common import config
# Optional override of the project root for outputs/results (used for offline fixture tests).
if os.environ.get("A1_ROOT_OVERRIDE"):
    config.ROOT_DIR = Path(os.environ["A1_ROOT_OVERRIDE"])

LIMIT = None  # set to an int for a smoke run; falls back to A1_LIMIT
LIMIT = config.resolve_limit(LIMIT)
OUT = config.out_root(LIMIT)
RES = config.results_root(LIMIT)
print("LIMIT =", LIMIT, "| outputs:", OUT, "| results:", RES)

LIMIT = None | outputs: /home/nguyen/School/CISC_451/assignment1/outputs | results: /home/nguyen/School/CISC_451/assignment1/results


In [2]:
RUNS = {  # run_name -> (label, model_id)
    "qwen25_7b": ("Qwen2.5-7B-Instruct", "qwen/qwen-2.5-7b-instruct"),
    "qwen25_72b": ("Qwen2.5-72B-Instruct", "qwen/qwen-2.5-72b-instruct"),
    "qwen3_32b_thinking": ("Qwen3-32B (thinking)", "qwen/qwen3-32b"),
}
from common import report, traces, env as env_mod
from common.preflight import PreflightReport

per_ex, model_ids = {}, {}
for run, (label, mid) in RUNS.items():
    p = OUT / run / "per_example.csv"
    if not p.is_file():
        print(f"[skip] {run}: {p} not found (run its notebook first)")
        continue
    per_ex[label] = pd.read_csv(p)
    model_ids[label] = mid
    print(f"[ok]   {run}: {len(per_ex[label])} rows")

[ok]   qwen25_7b: 1034 rows
[ok]   qwen25_72b: 1034 rows
[ok]   qwen3_32b_thinking: 1034 rows


## Model metadata

In [3]:
env_meta = env_mod.load_env(env_mod.env_path(LIMIT))
models = {mid: (env_meta.get("models", {}) or {}).get(mid, {}) for mid in model_ids.values()}
meta = report.metadata_table(models)
report.export_table(meta, "task1_metadata", "Model metadata: parameters, type, served providers "
                    "(call counts) and reported quantization.", RES)
meta

,model_id,params,type,quantization,provider,access
0,qwen/qwen-2.5-7b-instruct,7B,instruct,unknown,Phala (1034),API (OpenRouter)
1,qwen/qwen-2.5-72b-instruct,72B,instruct,fp8,DeepInfra (1034),API (OpenRouter)
2,qwen/qwen3-32b,32B,hybrid-thinking,fp8,DeepInfra (1034),API (OpenRouter)


## Accuracy (EX / EM by difficulty)

In [4]:
if per_ex:
    acc = report.accuracy_table(per_ex)
    report.export_table(acc, "task1_accuracy", "Execution accuracy (EX) and exact-set match (EM) "
                        "on Spider dev, overall and per difficulty, with example counts.", RES)
    display(acc)
else:
    print("No runs available.")

,model,n_easy,ex_easy,em_easy,n_medium,ex_medium,em_medium,n_hard,ex_hard,em_hard,n_extra,ex_extra,em_extra,n_all,ex_all,em_all
0,Qwen2.5-7B-Instruct,248,0.870968,0.520161,446,0.804933,0.282511,174,0.557471,0.224138,166,0.518072,0.066265,1034,0.733075,0.294971
1,Qwen2.5-72B-Instruct,248,0.903226,0.705645,446,0.878924,0.302691,174,0.724138,0.293103,166,0.584337,0.066265,1034,0.811412,0.359768
2,Qwen3-32B (thinking),248,0.899194,0.616935,446,0.827354,0.345291,174,0.706897,0.252874,166,0.566265,0.054217,1034,0.782398,0.348162


## Significance: bootstrap CIs and pairwise McNemar (EX)

In [5]:
if len(per_ex) >= 2:
    rng = np.random.default_rng(0)  # fixed seed for the bootstrap
    srt = {lab: df.sort_values("example_id").reset_index(drop=True) for lab, df in per_ex.items()}
    ci_rows = []
    for lab, df in srt.items():
        ex = df["ex"].to_numpy(dtype=float)
        bs = ex[rng.integers(0, len(ex), (2000, len(ex)))].mean(axis=1)
        lo, hi = np.percentile(bs, [2.5, 97.5])
        ci_rows.append({"model": lab, "n": len(ex), "ex_all": ex.mean(), "ci95_low": lo, "ci95_high": hi})
    ci = pd.DataFrame(ci_rows)
    report.export_table(ci, "task1_ex_ci", "Overall EX with 95% percentile bootstrap confidence "
                        "intervals (2,000 resamples of the 1,034 examples, seed 0).", RES)
    labs = list(srt); pairs = []
    for i in range(len(labs)):
        for j in range(i + 1, len(labs)):
            a, b = srt[labs[i]], srt[labs[j]]
            assert (a["example_id"].to_numpy() == b["example_id"].to_numpy()).all()
            m = report.mcnemar_exact(a["ex"], b["ex"])  # b: only model B correct, c: only A
            pairs.append({"model_a": labs[i], "model_b": labs[j], "ex_a": a["ex"].mean(),
                          "ex_b": b["ex"].mean(), "only_a_correct": m["c"],
                          "only_b_correct": m["b"], "mcnemar_p": m["p"]})
    pw = pd.DataFrame(pairs)
    pw["p_holm"] = traces.holm(pw["mcnemar_p"])
    report.export_table(pw, "task1_pairwise_mcnemar", "Pairwise exact McNemar tests on per-example "
                        "EX (same 1,034 examples); Holm-adjusted across the three pairs.", RES)
    display(ci); display(pw)

,model,n,ex_all,ci95_low,ci95_high
0,Qwen2.5-7B-Instruct,1034,0.733075,0.706963,0.761122
1,Qwen2.5-72B-Instruct,1034,0.811412,0.787234,0.834647
2,Qwen3-32B (thinking),1034,0.782398,0.759188,0.806576


,model_a,model_b,ex_a,ex_b,only_a_correct,only_b_correct,mcnemar_p,p_holm
0,Qwen2.5-7B-Instruct,Qwen2.5-72B-Instruct,0.733075,0.811412,44,125,3.474453e-10,1.042336e-09
1,Qwen2.5-7B-Instruct,Qwen3-32B (thinking),0.733075,0.782398,68,119,2.359431e-04,4.718862e-04
2,Qwen2.5-72B-Instruct,Qwen3-32B (thinking),0.811412,0.782398,82,52,1.195154e-02,1.195154e-02


## Efficiency and cost

In [6]:
if per_ex:
    eff = report.efficiency_table(per_ex, model_ids)
    report.export_table(eff, "task1_efficiency", "Mean latency and token usage per example, and total "
                        "cost. Token counts are provider-reported: mean_output_tokens is completion_tokens and "
                        "mean_reasoning_tokens is the provider's reasoning_tokens, which is an estimate on some providers "
                        "(DeepInfra) and can exceed completion_tokens; Task 2 measures trace length from the text. "
                        + report.price_note(list(model_ids.values())), RES)
    display(eff)

,model,n,mean_latency_s,mean_input_tokens,mean_output_tokens,mean_reasoning_tokens,total_cost_usd
0,Qwen2.5-7B-Instruct,1034,0.572613,836.974855,37.897485,0.000000,NaN
1,Qwen2.5-72B-Instruct,1034,1.912206,836.974855,34.371373,0.000000,NaN
2,Qwen3-32B (thinking),1034,30.998239,837.974855,981.472921,1023.610251,NaN


## Invalid outputs

In [7]:
if per_ex:
    inv = report.invalid_table(per_ex)
    report.export_table(inv, "task1_invalid", "Counts and rates of each SQL extraction status and of "
                        "non-executable predictions. unparseable = predictions the official Spider "
                        "parser cannot parse (scored EM=0 by the official script; EX unaffected); "
                        "em_parseable = EM on the parseable subset; ex_unparseable = EX on the "
                        "unparseable subset.", RES)
    display(inv)

,model,n,ok_count,ok_rate,fallback_count,fallback_rate,missing_count,missing_rate,truncated_count,truncated_rate,non_exec_count,non_exec_rate,unparseable_count,unparseable_rate,em_parseable,ex_unparseable
0,Qwen2.5-7B-Instruct,1034,1034,1.0,0,0.0,0,0.0,0,0.0,49,0.047389,568,0.549323,0.654506,0.654930
1,Qwen2.5-72B-Instruct,1034,1034,1.0,0,0.0,0,0.0,0,0.0,10,0.009671,510,0.493230,0.709924,0.725490
2,Qwen3-32B (thinking),1034,1034,1.0,0,0.0,0,0.0,0,0.0,6,0.005803,550,0.531915,0.743802,0.683636


## Environment and preflight

In [8]:
envt = report.environment_table(env_meta)
report.export_table(envt, "task1_environment", "Experimental environment: local machine (orchestration "
                    "and evaluation only), packages, and remote inference providers.", RES)
display(envt)

pf_path = OUT / "qwen3_32b_thinking" / "preflight.json"
if pf_path.is_file():
    pft = report.preflight_table(PreflightReport.load(pf_path))
    report.export_table(pft, "task1_preflight", "Qwen3-32B reasoning-trace preflight: served provider "
                        "and per-check results.", RES)
    display(pft)
else:
    print(f"[skip] preflight: {pf_path} not found")

,item,value
0,OS,Linux
1,Kernel,6.18.40.1-microsoft-standard-WSL2
2,Running under WSL,True
3,Distribution,Ubuntu 24.04.1 LTS
4,CPU,13th Gen Intel(R) Core(TM) i7-13620H
5,Logical cores,16
6,RAM (GiB),7.61
7,RAM note,RAM allocated to the WSL VM (from /proc/meminf...
8,Python,3.12.3
9,nltk version,3.10.3


,item,result,failing_ids
0,provider(s) served,DeepInfra,
1,example ids,"0, 1, 2, 3, 4, 5, 6, 7",
2,overall,PASS,
3,api_ok,PASS,
4,reasoning_nonempty,PASS,
5,reasoning_raw_not_summary,PASS,
6,reasoning_length_consistent,PASS,
7,reasoning_tokens_positive,PASS,
8,sql_extractable,PASS,
9,provider_matches_pin,PASS,


## EX by difficulty

In [9]:
if per_ex:
    levels = ["easy", "medium", "hard", "extra", "all"]
    fig, ax = plt.subplots(figsize=(8, 4))
    w = 0.8 / len(per_ex)
    x = np.arange(len(levels))
    for i, (label, row) in enumerate(acc.set_index("model").iterrows()):
        ax.bar(x + i * w - 0.4 + w / 2, [row[f"ex_{l}"] for l in levels], w, label=label)
    ax.set_xticks(x, levels); ax.set_ylim(0, 1); ax.set_ylabel("Execution accuracy")
    ax.set_title("EX by difficulty")
    ax.legend(fontsize=8, loc="upper left", bbox_to_anchor=(1.01, 1), borderaxespad=0)
    fig.tight_layout()
    print(report.save_figure(fig, "task1_ex_by_difficulty", RES))
    plt.close(fig)

{'png': PosixPath('/home/nguyen/School/CISC_451/assignment1/results/task1_ex_by_difficulty.png'), 'pdf': PosixPath('/home/nguyen/School/CISC_451/assignment1/results/task1_ex_by_difficulty.pdf')}
